# Module 7.1 — RAG Fusion

RAG Fusion = **Multi-query generation** + **Reciprocal Rank Fusion (RRF)**

**RRF score**: $\text{RRF}(d) = \sum_{r \in R} \frac{1}{k + r(d)}$  where k=60 (constant)

This merges ranked lists from multiple queries into a single, more comprehensive ranking.

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema import Document
from collections import defaultdict

docs = [
    Document(page_content="Quantum computing uses qubits that can exist in superposition."),
    Document(page_content="Quantum entanglement links qubits so measuring one instantly affects the other."),
    Document(page_content="Shor's algorithm on a quantum computer can factor large numbers exponentially faster."),
    Document(page_content="Current quantum computers suffer from decoherence and require near-absolute-zero temperatures."),
    Document(page_content="Classical computers use bits (0 or 1); quantum computers use qubits (0, 1, or both)."),
    Document(page_content="IBM and Google are leading quantum hardware manufacturers."),
]

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vs         = Chroma.from_documents(docs, embeddings, collection_name="fusion_demo")
llm        = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

# ── Step 1: Generate query variants ──────────────────────────────────────────
query_gen_prompt = ChatPromptTemplate.from_template("""
Generate {n} different search queries for: "{question}"
Output only the queries, one per line, no numbering.
""")

def generate_queries(question: str, n: int = 4) -> list[str]:
    chain   = query_gen_prompt | llm | StrOutputParser()
    result  = chain.invoke({"question": question, "n": n})
    queries = [q.strip() for q in result.strip().split("\n") if q.strip()]
    return queries[:n]

# ── Step 2: Reciprocal Rank Fusion ───────────────────────────────────────────
def reciprocal_rank_fusion(results_lists: list[list[Document]], k: int = 60) -> list[Document]:
    scores: dict[str, float]    = defaultdict(float)
    doc_map: dict[str, Document] = {}

    for ranked_list in results_lists:
        for rank, doc in enumerate(ranked_list, 1):
            key = doc.page_content
            scores[key]  += 1.0 / (k + rank)
            doc_map[key]  = doc

    sorted_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_map[key] for key, _ in sorted_docs]

# ── Step 3: Full RAG Fusion ───────────────────────────────────────────────────
original_query = "How do quantum computers work?"
variants       = generate_queries(original_query, n=4)
all_queries    = [original_query] + variants

print("Generated query variants:")
for q in all_queries:
    print(f"  • {q}")

retrieved_lists = [vs.similarity_search(q, k=4) for q in all_queries]
fused_docs      = reciprocal_rank_fusion(retrieved_lists)

print(f"\nRAG Fusion results ({len(fused_docs)} unique docs):")
for i, d in enumerate(fused_docs, 1):
    print(f"  [{i}] {d.page_content}")
